In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import accuracy_score
import joblib

print("Applying Recency Weighting to Master Ensemble...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
venue_intel = pd.read_csv('../data/processed/venue_intelligence.csv')

# Standardize teams
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad', 'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru', 'Gujarat Lions': 'Gujarat Titans'
}
matches.replace(team_mapping, inplace=True)
matches.dropna(subset=['winner', 'venue', 'date'], inplace=True)

# 2. Extract Year and Calculate Sample Weights
matches['date'] = pd.to_datetime(matches['date'])
matches['year'] = matches['date'].dt.year
max_year = matches['year'].max()

# If the match is within the last 2 years, weight = 3.0, else 1.0
matches['sample_weight'] = np.where(max_year - matches['year'] <= 2, 3.0, 1.0)

matches['target'] = (matches['team1'] == matches['winner']).astype(int)

# 3. Base Features
venue_win_dict = dict(zip(venue_intel['venue'], venue_intel['bat_first_win_pct']))
matches['venue_bat_first_win_pct'] = matches['venue'].map(venue_win_dict).fillna(50.0)

# Notice we include 'sample_weight' here temporarily
ml_df = matches[['team1', 'team2', 'venue', 'toss_decision', 'venue_bat_first_win_pct', 'target', 'sample_weight']].copy()

# 4. Train-Test Split (Chronological)
X = ml_df.drop('target', axis=1)
y = ml_df['target']

split_idx = int(len(ml_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 5. Separate the weights so they don't get fed into the model as a feature!
weights_train = X_train['sample_weight'].values
X_train = X_train.drop('sample_weight', axis=1)
X_test = X_test.drop('sample_weight', axis=1) # Test set doesn't need weights

# 6. Preprocessing
preprocessor = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), 
     ['team1', 'team2', 'venue', 'toss_decision'])
], remainder='passthrough')

# Transform features safely BEFORE fitting the model
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 7. Define Models & Ensemble
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('rf', model_rf), ('xgb', model_xgb)],
    voting='soft'
)

# 8. Train with Recency Weights!
print("Training Master Ensemble with 3x Recency Weights...")
ensemble.fit(X_train_transformed, y_train, sample_weight=weights_train)

# 9. Evaluate
y_pred = ensemble.predict(X_test_transformed)
accuracy = accuracy_score(y_test, y_pred)
print(f"🚀 Weighted Ensemble Accuracy: {accuracy * 100:.2f}%")

# 10. Save Model and Preprocessor Separately
os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(ensemble, '../models/weighted_ensemble.pkl')
joblib.dump(list(X_train.columns), '../models/ensemble_features.pkl')

print("✅ Weighted model and preprocessor saved successfully!")

Applying Recency Weighting to Master Ensemble...
Training Master Ensemble with 3x Recency Weights...


c:\Users\garvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


🚀 Weighted Ensemble Accuracy: 47.71%
✅ Weighted model and preprocessor saved successfully!
